Dataset source: https://www.data.gouv.fr/datasets/bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere-annees-de-2005-a-2024

# 01. Download raw data using the data.gouv.fr API

In [23]:
import requests
from pathlib import Path


def download_raw_data(
        year: int = 2021,
        datasets_api: str = "https://www.data.gouv.fr/api/1/datasets/",
        dataset_slug: str =  "bases-de-donnees-annuelles-des-accidents-corporels-de-la-circulation-routiere-annees-de-2005-a-2024/",
        raw_data_dir: str = "./data/raw"
        ) -> None:
    
    # API endpoint and output directory
    dataset_url = datasets_api + dataset_slug
    output_dir = Path(raw_data_dir)

    # Get dataset metadata
    response = requests.get(dataset_url)

    # Stop if API request was not successful
    if response.status_code != 200:
        raise Exception(f"Dataset API request failed ({response.status_code})")
    
    # Extract the list of resource metadata
    dataset = response.json()
    resources = dataset["resources"]

    # Search for the CSV files corresponding to the requested year
    for resource in resources:
        title = resource.get("title", "")
        url = resource.get("url", "")

        # Skip files that are non csv / from another year / are BAAC archives
        if not title.endswith(".csv") or str(year) not in title or "baac" in title:
            continue

        print(f"Downloading {title}...")
        destination = output_dir / title

        # Download file in chunks
        with requests.get(url, stream=True) as r:
            r.raise_for_status()
            with open(destination, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

In [24]:
years = [2021, 2022, 2023, 2024]

for year in years:
    download_raw_data(year=year, raw_data_dir="../../data/raw")

# 02. EDA

In [1]:
from collections import defaultdict
from pathlib import Path
import pandas as pd

raw_data_dir = Path("../../data/raw")

# Collect file paths ordered by file type + year
files = defaultdict(dict)
for file_path in raw_data_dir.glob("*.csv"):
    table = file_path.stem[:3].lower()
    year = int(file_path.stem[-4:])
    files[table][year] = file_path

# Display collected file paths
print("Files found in raw dir:")
for table, yearly_files in files.items():
    for year in sorted(yearly_files):
        print(year, yearly_files[year])

# Collect file metadata
metadata = {}
for table, yearly_files in files.items():
    metadata[table] = {}
    for year, path in sorted(yearly_files.items()):
        df = pd.read_csv(path, sep=";")
        metadata[table][year] = {
            "path": path,
            "shape": df.shape,
            "columns": list(df.columns),
            "dtypes": df.dtypes.astype(str).to_dict(),
            "missing": (100 * df.isna().mean()).round(2).to_dict(),
            "nunique": df.nunique(dropna=False).to_dict(),
            "head": df.head(),
        }

Files found in raw dir:
2021 ..\..\data\raw\carcteristiques-2021.csv
2022 ..\..\data\raw\carcteristiques-2022.csv
2023 ..\..\data\raw\caract-2023.csv
2024 ..\..\data\raw\Caract_2024.csv
2021 ..\..\data\raw\lieux-2021.csv
2022 ..\..\data\raw\lieux-2022.csv
2023 ..\..\data\raw\lieux-2023.csv
2024 ..\..\data\raw\Lieux_2024.csv
2021 ..\..\data\raw\usagers-2021.csv
2022 ..\..\data\raw\usagers-2022.csv
2023 ..\..\data\raw\usagers-2023.csv
2024 ..\..\data\raw\Usagers_2024.csv
2021 ..\..\data\raw\vehicules-2021.csv
2022 ..\..\data\raw\vehicules-2022.csv
2023 ..\..\data\raw\vehicules-2023.csv
2024 ..\..\data\raw\Vehicules_2024.csv


C:\Users\adria\AppData\Local\Temp\ipykernel_17264\3889897861.py:25: DtypeWarning: Columns (0: nbv) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, sep=";")
C:\Users\adria\AppData\Local\Temp\ipykernel_17264\3889897861.py:25: DtypeWarning: Columns (0: lartpc) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, sep=";")


In [58]:
# Create dataset overview
summary = []
for table, years in metadata.items():
    for year, info in years.items():
        summary.append({
            "table": table,
            "year": year,
            "rows": info["shape"][0],
            "columns": info["shape"][1],
        })

display(pd.DataFrame(summary))

,table,year,rows,columns
0,car,2021,56518,15
1,car,2022,55302,15
2,car,2023,54822,15
3,car,2024,54402,15
4,lie,2021,56518,18
5,lie,2022,55302,18
6,lie,2023,70860,18
7,lie,2024,70248,18
8,usa,2021,129248,16
9,usa,2022,126662,16


In [66]:
# Compare columns across years
for table, years in metadata.items():
    print(table + ":")

    all_columns = sorted(
        set().union(*(set(info["columns"]) for info in years.values()))
    )

    comparison = pd.DataFrame(index=all_columns)

    for year, info in years.items():
        comparison[year] = [
            col in info["columns"]
            for col in all_columns
        ]

    display(comparison.T)

car:


,Accident_Id,Num_Acc,adr,agg,an,atm,col,com,dep,hrmn,int,jour,lat,long,lum,mois
2021,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2022,True,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2023,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2024,False,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True


lie:


,Num_Acc,catr,circ,infra,larrout,lartpc,nbv,plan,pr,pr1,prof,situ,surf,v1,v2,vma,voie,vosp
2021,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2022,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2023,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2024,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True


usa:


,Num_Acc,actp,an_nais,catu,etatp,grav,id_usager,id_vehicule,locp,num_veh,place,secu1,secu2,secu3,sexe,trajet
2021,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2022,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2023,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True
2024,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True,True


veh:


,Num_Acc,catv,choc,id_vehicule,manv,motor,num_veh,obs,obsm,occutc,senc
2021,True,True,True,True,True,True,True,True,True,True,True
2022,True,True,True,True,True,True,True,True,True,True,True
2023,True,True,True,True,True,True,True,True,True,True,True
2024,True,True,True,True,True,True,True,True,True,True,True


In [68]:
# Compare dtypes
for table, years in metadata.items():

    print(table + ":")

    dtype_df = pd.DataFrame({
        year: info["dtypes"]
        for year, info in sorted(years.items())
    })

    display(dtype_df.T)

car:


,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,atm,col,adr,lat,long,Accident_Id
2021,int64,int64,int64,int64,str,int64,str,str,int64,int64,int64,int64,str,str,str,NaN
2022,NaN,int64,int64,int64,str,int64,str,str,int64,int64,int64,int64,str,str,str,int64
2023,int64,int64,int64,int64,str,int64,str,str,int64,int64,int64,int64,str,str,str,NaN
2024,int64,int64,int64,int64,str,int64,str,str,int64,int64,int64,int64,str,str,str,NaN


lie:


,Num_Acc,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
2021,int64,int64,str,int64,str,int64,int64,int64,int64,str,str,int64,str,str,int64,int64,int64,int64
2022,int64,int64,str,int64,str,int64,object,int64,int64,str,str,int64,str,str,int64,int64,int64,int64
2023,int64,int64,str,int64,str,int64,str,int64,int64,str,str,int64,object,str,int64,int64,int64,int64
2024,int64,int64,str,int64,str,int64,str,int64,int64,str,str,int64,str,str,int64,int64,int64,int64


usa:


,Num_Acc,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
2021,int64,str,str,str,int64,int64,int64,int64,float64,int64,int64,int64,int64,int64,str,int64
2022,int64,str,str,str,int64,int64,int64,int64,float64,int64,int64,int64,int64,int64,str,int64
2023,int64,str,str,str,int64,int64,int64,int64,float64,int64,int64,int64,int64,int64,str,int64
2024,int64,str,str,str,int64,int64,int64,int64,float64,int64,int64,int64,int64,int64,str,int64


veh:


,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
2021,int64,str,str,int64,int64,int64,int64,int64,int64,int64,float64
2022,int64,str,str,int64,int64,int64,int64,int64,int64,int64,float64
2023,int64,str,str,int64,int64,int64,int64,int64,int64,int64,float64
2024,int64,str,str,int64,int64,int64,int64,int64,int64,int64,float64


In [72]:
# Compare missing values
for table, years in metadata.items():

    print(table + ":")

    missing_df = pd.DataFrame({
        year: info["missing"]
        for year, info in sorted(years.items())
    })

    display(missing_df.T)

car:


,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,atm,col,adr,lat,long,Accident_Id
2021,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.01,0.0,0.0,NaN
2022,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.23,0.0,0.0,0.0
2023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.53,0.0,0.0,NaN
2024,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.25,0.0,0.0,NaN


lie:


,Num_Acc,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
2021,0.0,0.0,7.80,0.0,91.17,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.81,0.0,0.0,0.0,0.0,0.0
2022,0.0,0.0,8.69,0.0,90.50,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.95,0.0,0.0,0.0,0.0,0.0
2023,0.0,0.0,17.99,0.0,91.70,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.96,0.0,0.0,0.0,0.0,0.0
2024,0.0,0.0,18.98,0.0,91.58,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.95,0.0,0.0,0.0,0.0,0.0


usa:


,Num_Acc,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
2021,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.37,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.27,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.07,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2024,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.06,0.0,0.0,0.0,0.0,0.0,0.0,0.0


veh:


,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
2021,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.24
2022,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.14
2023,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,99.10
2024,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,98.98


In [73]:
# Compare uniques
for table, years in metadata.items():

    print(table + ":")

    nunique_df = pd.DataFrame({
        year: info["nunique"]
        for year, info in sorted(years.items())
    })

    display(nunique_df.T)

car:


,Num_Acc,jour,mois,an,hrmn,lum,dep,com,agg,int,atm,col,adr,lat,long,Accident_Id
2021,56518.0,31.0,12.0,1.0,1374.0,5.0,107.0,11150.0,2.0,9.0,10.0,8.0,29669.0,54618.0,54921.0,NaN
2022,NaN,31.0,12.0,1.0,1398.0,6.0,107.0,11253.0,2.0,10.0,10.0,8.0,29490.0,53457.0,53812.0,55302.0
2023,54822.0,31.0,12.0,1.0,1409.0,6.0,107.0,11311.0,2.0,10.0,10.0,8.0,29081.0,52905.0,53148.0,NaN
2024,54402.0,31.0,12.0,1.0,1414.0,5.0,107.0,11285.0,2.0,9.0,9.0,8.0,28872.0,52471.0,52742.0,NaN


lie:


,Num_Acc,catr,voie,v1,v2,circ,nbv,vosp,prof,pr,pr1,plan,lartpc,larrout,surf,infra,situ,vma
2021,56518,8,17264,4,27,5,14,5,5,457,1133,5,20,87,10,11,8,37
2022,55302,8,16311,4,26,5,29,5,5,463,1205,5,16,88,10,11,8,26
2023,54822,8,20064,4,25,5,15,5,5,446,1363,5,20,107,10,11,8,21
2024,54402,8,19751,4,28,5,15,5,5,461,1353,5,16,65,10,11,8,36


usa:


,Num_Acc,id_usager,id_vehicule,num_veh,place,catu,grav,sexe,an_nais,trajet,secu1,secu2,secu3,locp,actp,etatp
2021,56518,129248,97309,53,11,3,5,3,105,8,11,11,9,11,13,4
2022,55302,126662,94493,47,11,3,5,3,105,8,11,11,11,11,13,4
2023,54822,125789,93545,46,11,3,5,3,108,8,11,11,11,11,13,4
2024,54402,125187,92654,45,11,3,4,3,106,8,11,11,11,11,13,4


veh:


,Num_Acc,id_vehicule,num_veh,senc,catv,obs,obsm,choc,manv,motor,occutc
2021,56518,97315,53,5,32,19,8,11,28,8,19
2022,55302,94493,47,5,32,19,8,11,28,8,24
2023,54822,93585,46,5,32,19,8,11,28,8,28
2024,54402,92678,45,5,32,19,8,11,28,8,24


# 03. Write preprocessing pipeline

In [ ]:
from collections import defaultdict
from pathlib import Path

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer



def process_data(
    years: list[int] = [2021, 2022, 2023, 2024],
    exclusive_test_year: int | None = None,
    raw_data_dir: str = "./data/raw",
    process_data_dir: str = "./data/processed",
    normalize: bool = True
    ) -> None:

  
    raw_paths = collect_raw_data_paths(raw_data_dir=raw_data_dir)

    #--Preprocess data
    df_years = {}
    for year in years:
        df_years[year] = process_yearly_data(
            path_collection=raw_paths,
            year=year)

    #--Split data      
    X_train, X_test, y_train, y_test = split_data(
        df_collection=df_years,
        exclusive_test_year=exclusive_test_year)
    


    #--Filling NaN values
    col_to_fill_na = ["surf", "circ", "col", "motor"]
    X_train[col_to_fill_na] = X_train[col_to_fill_na].fillna(X_train[col_to_fill_na].mode().iloc[0])
    X_test[col_to_fill_na] = X_test[col_to_fill_na].fillna(X_train[col_to_fill_na].mode().iloc[0])

    #--Get rid of non-breaking spaces
    obj_cols = X_train.select_dtypes(include="object").columns
    X_train[obj_cols] = X_train[obj_cols].apply(
        lambda s: s.str.replace("\xa0", "", regex=False).str.strip()    
    )
    obj_cols = X_test.select_dtypes(include="object").columns
    X_test[obj_cols] = X_test[obj_cols].apply(
        lambda s: s.str.replace("\xa0", "", regex=False).str.strip()    
    )

    #--Replace NaN values with medians
    imputer = SimpleImputer(strategy="median")
    
    X_train = pd.DataFrame(
        imputer.fit_transform(X_train),
        columns=X_train.columns,
        index=X_train.index
    )

    X_test = pd.DataFrame(
        imputer.transform(X_test),
        columns=X_test.columns,
        index=X_test.index
    )


    #--Normalize features
    if normalize:
        scaler = StandardScaler()

        X_train = pd.DataFrame(
            scaler.fit_transform(X_train),
            columns=X_train.columns,
            index=X_train.index,
        )

        X_test = pd.DataFrame(
            scaler.transform(X_test),
            columns=X_test.columns,
            index=X_test.index,
        )

    #--Saving preprocessed data
    for file, filename in zip([X_train, X_test, y_train, y_test], ['X_train', 'X_test', 'y_train', 'y_test']):
        out_path = Path(process_data_dir) / f"{filename}.csv"
        file.to_csv(out_path, index=False)


def process_yearly_data(
        path_collection: defaultdict,
        year: int
        ) -> pd.DataFrame:
    
    #--Importing dataset
    df_users = pd.read_csv(path_collection["usa"][year], sep=";")
    df_caract = pd.read_csv(path_collection["car"][year], sep=";")
    df_places = pd.read_csv(path_collection["lie"][year], sep=";")
    df_veh = pd.read_csv(path_collection["veh"][year], sep=";")

    #--Correct accident ID annomaly
    df_users = correct_id_anomaly(df_users)
    df_caract = correct_id_anomaly(df_caract)
    df_places = correct_id_anomaly(df_places)
    df_veh = correct_id_anomaly(df_veh)

    #--Creating new columns
    nb_victim = pd.crosstab(df_users.Num_Acc, "count").reset_index()
    nb_vehicules = pd.crosstab(df_veh.Num_Acc, "count").reset_index()
    df_users["year_acc"] = df_users["Num_Acc"].astype(str).apply(lambda x : x[:4]).astype(int)
    df_users["victim_age"] = df_users["year_acc"]-df_users["an_nais"]
    for i in df_users["victim_age"] :
        if (i>120)|(i<0):
            df_users["victim_age"].replace(i,np.nan)
    df_caract["hour"] = df_caract["hrmn"].astype(str).apply(lambda x : x[:-3])
    df_caract.drop(['hrmn', 'an'], inplace=True, axis=1)
    df_users.drop(['an_nais'], inplace=True, axis=1)

    #--Replacing names 
    df_users.grav.replace([1,2,3,4], [1,3,4,2], inplace = True)
    df_caract.rename({"agg" : "agg_"},  inplace = True, axis = 1)
    df_caract["dep"] = df_caract["dep"].str.replace("2A", "201")
    df_caract["dep"] = df_caract["dep"].str.replace("2B", "202")
    df_caract["com"] = df_caract["com"].str.replace("2A", "201")
    df_caract["com"] = df_caract["com"].str.replace("2B", "202")

    #--Converting columns types
    cols = ["dep", "com", "hour"]
    df_caract[cols] = df_caract[cols].apply(
        pd.to_numeric,
        errors="coerce"
    ).astype("Int64")

    dico_to_float = { 'lat': float, 'long':float}
    df_caract["lat"] = df_caract["lat"].str.replace(',', '.')
    df_caract["long"] = df_caract["long"].str.replace(',', '.')
    df_caract = df_caract.astype(dico_to_float)

    #--Grouping modalities 
    dico = {1:0, 2:1, 3:1, 4:1, 5:1, 6:1,7:1, 8:0, 9:0}
    df_caract["atm"] = df_caract["atm"].replace(dico)
    catv_value = [0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,30,31,32,33,34,35,36,37,38,39,40,41,42,43,50,60,80,99]
    catv_value_new = [0,1,1,2,1,1,6,2,5,5,5,5,5,4,4,4,4,4,3,3,4,4,1,1,1,1,1,6,6,3,3,3,3,1,1,1,1,1,0,0]
    df_veh['catv'].replace(catv_value, catv_value_new, inplace = True)

    #--Merging datasets 
    fusion1= df_users.merge(df_veh, on = ["Num_Acc","num_veh", "id_vehicule"], how="inner")
    fusion1 = fusion1.sort_values(by = "grav", ascending = False)
    fusion1 = fusion1.drop_duplicates(subset = ['Num_Acc'], keep="first")
    fusion2 = fusion1.merge(df_places, on = "Num_Acc", how = "left")
    df = fusion2.merge(df_caract, on = 'Num_Acc', how="left")

    #--Adding new columns
    df = df.merge(nb_victim, on = "Num_Acc", how = "inner")
    df.rename({"count" :"nb_victim"},axis = 1, inplace = True) 
    df = df.merge(nb_vehicules, on = "Num_Acc", how = "inner") 
    df.rename({"count" :"nb_vehicules"},axis = 1, inplace = True)

    #--Modification of the target variable  : 1 : prioritary // 0 : non-prioritary
    df['grav'].replace([2,3,4], [0,1,1], inplace=True)

    #--Replacing values -1 and 0 
    col_to_replace0_na = [ "trajet", "catv", "motor"]
    col_to_replace1_na = [ "trajet", "secu1", "catv", "obsm", "motor", "circ", "surf", "situ", "vma", "atm", "col"]
    df[col_to_replace1_na] = df[col_to_replace1_na].replace(-1, np.nan)
    df[col_to_replace0_na] = df[col_to_replace0_na].replace(0, np.nan)

    #--Dropping columns 
    list_to_drop = ['senc','larrout','actp', 'manv', 'choc', 'nbv', 'prof', 'plan', 'Num_Acc', 'id_vehicule', 'num_veh', 'pr', 'pr1','voie', 'trajet',"secu2", "secu3",'adr', 'v1', 'lartpc','occutc','v2','vosp','locp','etatp', 'infra', 'obs' ]
    df.drop(list_to_drop, axis=1, inplace=True)

    #--Dropping lines with NaN values
    col_to_drop_lines = ['catv', 'vma', 'secu1', 'obsm', 'atm']
    df = df.dropna(subset = col_to_drop_lines, axis=0)

    return df


def split_data(
        df_collection: dict[int, pd.DataFrame],
        exclusive_test_year: int | None = None
        ):
    
    if exclusive_test_year:
        train = pd.concat(
            [df for year, df in df_collection.items()
            if year != exclusive_test_year],
            ignore_index=True,
        )

        X_train = train.drop(["grav"], axis = 1)
        y_train = train["grav"]

        test = df_collection[exclusive_test_year]

        X_test = test.drop(["grav"], axis = 1)
        y_test = test["grav"]

    else:
        df = pd.concat(
            [df for year, df in df_collection.items()
            if year != exclusive_test_year],
            ignore_index=True,
        )

        target = df['grav']
        feats = df.drop(['grav'], axis = 1)

        X_train, X_test, y_train, y_test = train_test_split(
            feats, target, test_size=0.3, random_state = 42
            )
        
    return X_train, X_test, y_train, y_test



def collect_raw_data_paths(
        raw_data_dir: str = "./data/raw"
        ) -> defaultdict:
    # Collect file paths ordered by file type + year
    raw_data_dir = Path(raw_data_dir)
    files = defaultdict(dict)
    for file_path in raw_data_dir.glob("*.csv"):
        table = file_path.stem[:3].lower()
        year = int(file_path.stem[-4:])
        files[table][year] = file_path
    
    return files

def correct_id_anomaly(df: pd.DataFrame) -> pd.DataFrame:
    if df.get("Accident_Id", None) is not None:
        df["Num_Acc"] = df.Accident_Id
        df = df.drop(columns=["Accident_Id"])
    return df


process_data(raw_data_dir="../../data/raw",
             process_data_dir= "../../data/processed",
             years=[2021, 2022, 2023])

C:\Users\adria\AppData\Local\Temp\ipykernel_2356\4291933230.py:120: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df_users.grav.replace([1,2,3,4], [1,3,4,2], inplace = True)
C:\Users\adria\AppData\Local\Temp\ipykernel_2356\4291933230.py:144: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chain

# 04. Experiment with different models on preprocessed data

In [8]:
import time
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier,
    AdaBoostClassifier,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# ============================================================
# Load data
# ============================================================

processed_data_dir = "../../data/processed/"
X = pd.read_csv(processed_data_dir + "X_train.csv")
y = pd.read_csv(processed_data_dir + "y_train.csv").squeeze()


le = LabelEncoder()
y = le.fit_transform(y)


# ============================================================
# Train / Validation split
# ============================================================

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

print(f"Training samples:   {len(X_train):,}")
print(f"Validation samples: {len(X_val):,}")

# ============================================================
# Models
# ============================================================

models = {

    "Logistic Regression":
        LogisticRegression(
            max_iter=1000,
            random_state=42,
        ),

    "Linear SVM":
        LinearSVC(
            max_iter=5000,
            random_state=42,
        ),

    "KNN":
        KNeighborsClassifier(
            n_neighbors=5,
        ),

    "Decision Tree":
        DecisionTreeClassifier(
            random_state=42,
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=200,
            n_jobs=-1,
            random_state=42,
        ),

    "Extra Trees":
        ExtraTreesClassifier(
            n_estimators=200,
            n_jobs=-1,
            random_state=42,
        ),

    "HistGradientBoosting":
        HistGradientBoostingClassifier(
            random_state=42,
        ),

    "AdaBoost":
        AdaBoostClassifier(
            random_state=42,
        ),

    "Gaussian Naive Bayes":
        GaussianNB(),

    "Neural Network":
        MLPClassifier(
            hidden_layer_sizes=(100,),
            max_iter=300,
            early_stopping=True,
            random_state=42,
        ),

    "XGBoost":
        XGBClassifier(
            n_estimators=200,
            learning_rate=0.1,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            tree_method="hist",
            eval_metric="logloss",
            random_state=42,
        ),

    "LightGBM":
        LGBMClassifier(
            n_estimators=200,
            learning_rate=0.1,
            random_state=42,
            verbose=-1,
        )
}

# ============================================================
# Benchmark
# ============================================================

results = []

trained_models = {}

for name, model in models.items():

    print("="*40)
    print("Fitting data to", name)

    start = time.time()

    model.fit(X_train, y_train)

    train_time = time.time() - start

    y_pred = model.predict(X_val)

    accuracy = accuracy_score(y_val, y_pred)

    results.append({
        "Model": name,
        "Accuracy": accuracy,
        "Training Time (s)": f"{train_time:3.2f}",
    })

    trained_models[name] = model

    stop = time.time()
    print(f"Done. Iteration took {stop-start:3.2f}s")

# ============================================================
# Results
# ============================================================

results_df = (
    pd.DataFrame(results)
      .sort_values("Accuracy", ascending=False)
      .reset_index(drop=True)
)

print("\n")
print(results_df)

#results_df.to_csv("model_comparison.csv", index=False)

print("\nBest model:")
print(results_df.iloc[0])

Training samples:   98,313
Validation samples: 24,579
Fitting data to Logistic Regression
Done. Iteration took 0.81s
Fitting data to Linear SVM
Done. Iteration took 2.86s
Fitting data to KNN
Done. Iteration took 5.94s
Fitting data to Decision Tree
Done. Iteration took 2.49s
Fitting data to Random Forest
Done. Iteration took 13.81s
Fitting data to Extra Trees
Done. Iteration took 13.14s
Fitting data to HistGradientBoosting
Done. Iteration took 5.83s
Fitting data to AdaBoost
Done. Iteration took 10.09s
Fitting data to Gaussian Naive Bayes
Done. Iteration took 0.12s
Fitting data to Neural Network
Done. Iteration took 8.26s
Fitting data to XGBoost
Done. Iteration took 10.63s
Fitting data to LightGBM
Done. Iteration took 7.70s


                   Model  Accuracy Training Time (s)
0          Random Forest  0.778144             13.47
1                XGBoost  0.771594             10.48
2            Extra Trees  0.770902             12.49
3               LightGBM  0.769926              7.31
4

# 05. Optional: Clean produced data files and artifacts

In [ ]:

from pathlib import Path
directory = Path("../../data/raw")

for csv_file in directory.glob("*.csv"):
    csv_file.unlink()
    print(f"Deleted: {csv_file}")

In [ ]:
from pathlib import Path
directory = Path("../../data/processed")

for csv_file in directory.glob("*.csv"):
    csv_file.unlink()
    print(f"Deleted: {csv_file}")